# **CCB02. Predicting subcellular localizations of *Cyclospora cayetanensis* proteins using DeepLoc2.1.**

**Date**: Oct 10, 2026

**TL;DR**
- **Goal**: Determine the localizations of the C. cayetanensis (CCay) proteome to find a strong candidate for a rapid antigen test target (with a focus on extracellular proteins).
- **Problem**: There is little to no protein localization data (MS, mRNA, etc.) for CCay to use as reference.
- **Solution**: Use DeepLoc2.1, a deep learning model that predicts protein localization, to shortlist a set of targets.

---

**Reads**: `data/reference/protein.faa` — the CcayRef3 reference proteome

**Writes**: `results/localization.tsv` — per-protein localization predictions, membrane type, and per-compartment probability scores, renamed from DeepLoc's timestamped output (`results_20261008-181607.csv`)

---

**Resources used**:

- NCBI Datasets, C. cayetanensis proteome.
  - O'Leary NA, Cox E, Holmes JB, Anderson WR, Falk R, Hem V, Tsuchiya MTN, Schuler GD, Zhang X, Torcivia J, Ketter A, Breen L, Cothran J, Bajwa H, Tinne J, Meric PA, Hlavina W, Schneider VA. Exploring and retrieving sequence and metadata for species across the tree of life with NCBI Datasets. Sci Data. 2024 Jul 5;11(1):732. doi: 10.1038/s41597-024-03571-y. PMID: 38969627; PMCID: PMC11226681.
- [DeepLoc2.1](https://services.healthtech.dtu.dk/services/DeepLoc-2.1/)
  - DeepLoc 2.1: multi-label membrane protein type prediction using protein language models. Marius Thrane Ødum, Felix Teufel, Vineet Thumuluri, José Juan Almagro Armenteros, Alexander Rosenberg Johansen, Ole Winther, Henrik Nielsen. Nucleic Acids Research, Web server issue 2024.
  - Note: the DeepLoc2.1 webserver is free to use in batches of 100 proteins. This notebook uses the downloadable package for academic users, which you can request [here](https://services.healthtech.dtu.dk/cgi-bin/sw_request?software=deeploc&version=2.1&packageversion=2.1&platform=All).

## **0. Imports & installs.**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import pandas as pd

In [ ]:
! curl -o datasets 'https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets'
! curl -o dataformat 'https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/dataformat'
! chmod +x datasets datasets
! chmod +x datasets dataformat

In [ ]:
! ./datasets download genome taxon 88456  --include all --filename ccay_genomes.zip

In [ ]:
! unzip ccay_genomes.zip

**⬆ (Upload your copy of the DeepLoc2.1 tar archive)**

In [ ]:
! tar -xzf deeploc-2.1.All.tar.gz
! cd deeploc2_package/ && pip install .

In [ ]:
! deeploc2 -f /content/ncbi_dataset/data/GCF_002999335.1/protein.faa -m Fast -o /content/deeploc2_output/ -d cuda

^C


In [ ]:
localizations = pd.read_csv('/content/deeploc2_output/results_20261008-181607.csv')

In [ ]:
localizations

,Protein_ID,Localizations,Signals,Membrane types,Cytoplasm,Nucleus,Extracellular,Cell membrane,Mitochondrion,Plastid,Endoplasmic reticulum,Lysosome/Vacuole,Golgi apparatus,Peroxisome,Peripheral,Transmembrane,Lipid anchor,Soluble
0,XP_022586349.2,Endoplasmic reticulum,Signal peptide|Transmembrane domain,Transmembrane,0.1282,0.1820,0.0449,0.3204,0.1403,0.0166,0.8768,0.3592,0.4063,0.0034,0.0602,0.9968,0.0733,0.0925
1,XP_022586352.2,Cytoplasm,Nuclear localization signal,Soluble,0.6825,0.5030,0.0695,0.2695,0.1132,0.0119,0.0488,0.1567,0.2283,0.0216,0.5857,0.0544,0.0791,0.7674
2,XP_022586359.2,Cytoplasm,Nuclear export signal,Soluble,0.5634,0.4525,0.0260,0.2451,0.3105,0.0432,0.1402,0.3217,0.3660,0.0141,0.6064,0.0998,0.0761,0.6501
3,XP_022586363.2,Endoplasmic reticulum,Signal peptide,Soluble,0.1384,0.0863,0.5286,0.1683,0.1092,0.0751,0.5906,0.5054,0.4678,0.0087,0.4487,0.1352,0.0294,0.6851
4,XP_022586398.1,Cytoplasm|Nucleus,NaN,Soluble,0.5415,0.5831,0.2129,0.0653,0.2987,0.0305,0.1886,0.0914,0.2443,0.0206,0.5487,0.0857,0.1642,0.7805
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5788,XP_026194603.1,Mitochondrion,Mitochondrial transit peptide,Soluble,0.4235,0.2966,0.2417,0.1441,0.7741,0.1214,0.1765,0.1158,0.2302,0.0267,0.6051,0.1319,0.0687,0.6173
5789,XP_026194604.1,Cytoplasm,NaN,Soluble,0.4804,0.3278,0.2879,0.1129,0.3103,0.0687,0.1284,0.0544,0.5242,0.0153,0.6054,0.1531,0.0870,0.6940
5790,XP_026194605.1,Extracellular,Signal peptide,Peripheral|Soluble,0.3379,0.1956,0.5913,0.2018,0.2659,0.2069,0.0863,0.2397,0.5274,0.0142,0.6715,0.0787,0.0400,0.7199
5791,XP_026194606.1,Extracellular|Golgi apparatus,Signal peptide,Soluble,0.1699,0.1170,0.7562,0.3435,0.0718,0.0397,0.4474,0.3616,0.6840,0.0019,0.3082,0.2178,0.1171,0.6940
